# 2. Regression — Predicting `target_regression`
**APRAU 2026/27 — Group 18**

Sections:
1. Setup & Data Loading
2. Preprocessing
3. Simple Linear Regression
4. Multiple Linear Regression
5. Comparison & Discussion

## 1. Setup

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_PATH = '/content/drive/MyDrive/APRAU/dataset_grupo_18.csv'
else:
    DATASET_PATH = '../dataset_grupo_18.csv'

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler, LabelEncoder

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42

## 2. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Encode categorical columns
df_enc = df.copy()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = LabelEncoder().fit_transform(df_enc[col].astype(str))

TARGET = 'target_regression'
FEATURES = [c for c in df_enc.columns if c not in [TARGET, 'target_class']]

X = df_enc[FEATURES]
y = df_enc[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
def evaluate(name, y_true, y_pred):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f'[{name}]  R²={r2:.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}')
    return {'Model': name, 'R²': r2, 'MAE': mae, 'RMSE': rmse}

## 3. Simple Linear Regression

In [ ]:
# Test each feature individually and rank by R²
results_simple = []
for feat in FEATURES:
    model = LinearRegression()
    model.fit(X_train[[feat]], y_train)
    y_pred = model.predict(X_test[[feat]])
    r2 = r2_score(y_test, y_pred)
    results_simple.append({'Feature': feat, 'R²': r2})

results_simple = pd.DataFrame(results_simple).sort_values('R²', ascending=False)
results_simple.head(10)

In [ ]:
# Best single feature model
best_feat = results_simple.iloc[0]['Feature']
print(f'Best single feature: {best_feat}')

model_slr = LinearRegression()
model_slr.fit(X_train[[best_feat]], y_train)
y_pred_slr = model_slr.predict(X_test[[best_feat]])

metrics = [evaluate('Simple LR', y_test, y_pred_slr)]

plt.scatter(X_test[best_feat], y_test, alpha=0.4, label='Actual')
plt.scatter(X_test[best_feat], y_pred_slr, alpha=0.4, label='Predicted')
plt.xlabel(best_feat)
plt.ylabel('target_regression')
plt.legend()
plt.title(f'Simple LR — {best_feat}')
plt.tight_layout()
plt.show()

## 4. Multiple Linear Regression

In [ ]:
# All features
model_mlr_all = LinearRegression()
model_mlr_all.fit(X_train, y_train)
y_pred_mlr_all = model_mlr_all.predict(X_test)
metrics.append(evaluate('Multiple LR (all features)', y_test, y_pred_mlr_all))

In [ ]:
# Top N features (from Simple LR ranking)
TOP_N = 5
top_feats = results_simple.head(TOP_N)['Feature'].tolist()
print(f'Top {TOP_N} features: {top_feats}')

model_mlr_top = LinearRegression()
model_mlr_top.fit(X_train[top_feats], y_train)
y_pred_mlr_top = model_mlr_top.predict(X_test[top_feats])
metrics.append(evaluate(f'Multiple LR (top {TOP_N})', y_test, y_pred_mlr_top))

## 5. Comparison & Discussion

In [ ]:
metrics_df = pd.DataFrame(metrics).set_index('Model')
metrics_df

In [ ]:
metrics_df[['R²', 'MAE', 'RMSE']].plot(kind='bar', figsize=(10, 5))
plt.title('Model Comparison')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Discussion

*(Fill in after running the models)*

- **Best single feature and justification:**
- **Best feature combination and justification:**
- **Simple vs Multiple LR comparison:**